In [1]:
import os

In [2]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow\\notebooks'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow'

In [5]:
## Preparing entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    transformed_data_path: Path
    preprocessor_obj_file_path: Path
    transformed_data_obj_file_path: Path
    TRANSFORMATION_REPORT: Path

In [6]:
## Configuration

from wdmproject.constants import *
from wdmproject.utils.common import read_yaml, create_directories

In [8]:
## Configuration manager class
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])

    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            transformed_data_path=Path(config.transformed_data_path),
            preprocessor_obj_file_path=Path(config.preprocessor_obj_file_path),
            transformed_data_obj_file_path=Path(config.transformed_data_obj_file_path),
            TRANSFORMATION_REPORT=Path(config.TRANSFORMATION_REPORT)
        )

        return data_transformation_config

In [9]:
## Component

import os
import json
from wdmproject import logger
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.decomposition import PCA
import pickle, joblib



In [10]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.data = None
        self.transformation_report = {}


    ## Load Dataset for transformation
    def load_data(self):
        try:
            self.data = pd.read_excel(self.config.data_path)
            #with open(self.config.STATUS_FILE, 'w') as f:
            #       f.write(f"Data Loaded Successfully. \n")
            logger.info('Data Loaded Succesfully.')

            self.transformation_report['dataset_load'] = {
                 "status" : "Success",
                 "rows" : self.data.shape[0],
                 "columns" : self.data.shape[1]
            }

                  

        except Exception as e:
            logger.error(f"Error Loading Dataset. {e}")
            self.transformation_report['dataset_load'] = {
                 "status" : "Failed",
                 "error_msg" : "Error Loading Dataset.",
                 "error" : str(e),
            }
            raise e
        

    ## Droping and Saving Country Column for later use

    def separate_country_column(self):

        self.country = self.data["country"].reset_index(drop=True)
        
        self.data = self.data.drop(columns=["country", "number_of_records"]).reset_index(drop=True)

        logger.info("Country column separated.")

    
    ## Removing currency symbols and any other spaces from the data
    # Data cleaning and converting object dtype to numeric

    def convert_object_to_numeric(self):
        try:
            data = self.data.copy()

            logger.info("Cleaning and validating the data before preprocessing.")

            data.replace([" ","NA","N/A","null","None",""], np.nan, inplace=True)

            for col in data.columns:
                if data[col].dtype == "object":

                    data[col] = (
                        data[col]
                        .astype(str)
                        .str.replace(r"[^\d.-]","",regex=True)
                    )

                    data[col] = pd.to_numeric(data[col], errors='coerce').astype('float64')

            #dt = data.info()
            logger.info(f"Object columns converted to numeric.")
            
            inf_count = np.isinf(data.values).sum()
            if inf_count > 0:
                logger.warning(f"Replacing {inf_count} infinite values with NaN")
                data.replace([np.inf, -np.inf], np.nan, inplace=True)

            empty_cols = data.columns[data.isna().all()].tolist()
            if empty_cols:
                logger.warning(f"Dropping fully empty columns: {empty_cols}")
                data.drop(columns=empty_cols, inplace=True)

            total_missing = data.isna().sum().sum()
            logger.info(f"Total missing values after cleaning: {total_missing}") 

            logger.info(f"Data types after conversion:\n{data.dtypes}")
            
            self.data = data

            logger.info(f"Dataset Shape after cleaning: {data.shape}")

            return data   

    
        except Exception as e:
            logger.info(f"Object columns conversion to numeric failed : {e}")
            raise e
    

    ## Creating numeric Pipeline preprocessor with Imputer, power transformer and Roburst scaler 

    def create_preprocessor(self):
        try:
            data = self.data
            
            logger.info("Creating preprocessing pipeline...")
            numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns.tolist()
            
            logger.info(f"Total numeric features for transformation: {len(numeric_cols)}")
            logger.info(f"Numeric features: {numeric_cols}")

            skewed_cols = data[numeric_cols].skew()
            log_cols = skewed_cols[skewed_cols > 1].index.tolist()
            non_log_cols = [col for col in numeric_cols if col not in log_cols]

            # check problematic values in log columns
            for col in log_cols:
                col_min = data[col].min()

                if col_min <= -1:
                    print(f"{col} has values <= -1 -> min: {col_min}")

            ## for skewed columns appying log
            log_pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("log_transform", FunctionTransformer(np.log1p)),
                ("scaler", StandardScaler())
            ])

            ## For normal columns (no log)
            numeric_pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())                    
            ])

            logger.info("Numeric pipeline and log pipeline created with steps:")
            logger.info("1. SimpleImputer (strategy='median')")
            logger.info("2. LogTransformer (np.log1p)")
            logger.info("3. StandardScaler")

            logger.info(f"Type of numeric_pipeline: {type(numeric_pipeline)}")
            
            preprocessor = ColumnTransformer(
                transformers=[
                    ("log_pipeline", log_pipeline, log_cols),
                    ("num_pipeline", numeric_pipeline, non_log_cols)
                ]
            )

            logger.info("ColumnTransformer created successfully.")

            self.transformation_report["preprocessing"] = {
                "status": "Success",
                "total_numeric_features": len(numeric_cols),
                "numeric_features": numeric_cols,
                "pipeline_steps": [
                    "SimpleImputer",
                    "LogTransformer (log1p)",
                    "StandardScaler"
                ]
            }

            return preprocessor

        except Exception as e:
            logger.error(f"Preprocessor creation failed: {e}")

            self.transformation_report["preprocessing"] = {
                "status": "Failed",
                "error": str(e)
            }

            raise e
        

    ## Creating Full Pipeline with PCA
    def create_full_pipeline(self):
        try:
            preprocessor = self.create_preprocessor()

            logger.info("Applying PCA with 90% variance retention")

            pca = PCA(n_components=0.90, random_state=42)

            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("pca", pca)
            ])

            logger.info("Pipeline with PCA created successfully")

            self.transformation_report["pca"] = {
                "status": "Success",
                "variance_retained": 0.90,
                "random_state": 42
            }

            return pipeline
        
        except Exception as e:

            logger.error(f"Preprocessor with PCA creation failed: {e}")

            self.transformation_report["pca"] = {
                "status": "Failed",
                "error": str(e)
            }

            raise e
        

    ## Transform Data fit data function
    def transform_data(self):
        try:
            data = self.data

            pipeline = self.create_full_pipeline()

            logger.info(f"Fitting Transformation Pipiline.")

            transformed_data = pipeline.fit_transform(data)

            logger.info("Transformation completed successfully")
           
            ## Check KNN Imputation
            missing_before = data.isnull().sum().sum()
            logger.info(f"Total missing values before imputation: {missing_before}")
            missing_after = np.isnan(transformed_data).sum()
            logger.info(f"Total missing values after transformation: {missing_after}")

            self.transformation_report["preprocessing"]["imputation"] = {
                "method": "SimpleImputer",
                "strategy": "median",
                "missing_before": int(missing_before),
                "missing_after": int(missing_after)
            }

            ## Checking Power Transformations

            skew_before = self.data.skew().to_dict()
            df_transformed = pd.DataFrame(transformed_data)
            skew_after = df_transformed.skew().to_dict()

            logger.info("Skewness before transformation calculated")
            logger.info("Skewness after transformation calculated")

            self.transformation_report["preprocessing"]["log_transformation"] = {
                "method": "log1p",
                "skewness_before_sample": dict(list(skew_before.items())[:5]),
                "skewness_after_sample": dict(list(skew_after.items())[:5])
            }
            
            ## Checking Scaling Effect
            logger.info("Feature scaling completed using RobustScaler")

            self.transformation_report["preprocessing"]["scaling"] = {
                "method": "StandardScaler",
                "sample_feature_stats_after_scaling": {
                    "mean": df_transformed.mean().head().to_dict(),
                    "std": df_transformed.std().head().to_dict()
                }
            }

            ## PCA
            pca = pipeline.named_steps["pca"]
            
            logger.info(f"PCA Components: {pca.n_components_}")
            logger.info(f"Explained Variance Ratio: {pca.explained_variance_ratio_}")

            self.transformation_report["pca"] = {
                "status": "Success",
                "components_generated": int(pca.n_components_),
                "variance_ratio": pca.explained_variance_ratio_.tolist(),
                "random_state": 42 
            }

            logger.info(f"Data Transformation pipeline Completed Successfully")
            
            return transformed_data, pipeline

        except Exception as e:
            logger.error(f"Transformation pipeline failed : {str(e)}")
            raise e     


    ## Saving transformed Dataset
    def save_transformed_data(self, transformed_data):
        try: 

            ## Sometimes power transformers creates nan for extreme values, if present, converting to zero
           
            df_transformed = pd.DataFrame(transformed_data)
            #df_transformed = pd.DataFrame(transformed_data).fillna(0).values
            df_transformed["country"] = self.country.values

            transformed_data_path = self.config.transformed_data_path
            self.data.to_excel(transformed_data_path, index=False)

            joblib.dump(
                transformed_data,
                self.config.transformed_data_obj_file_path
            )


            logger.info(f"Transformed dataset saved at : {transformed_data_path}")
        except Exception as e:
            logger.error(f"Saving transformed dataset failed: {e}")
            raise e
        

    ## Saving Preprocessor Object
    def save_preprocessor(self, pipeline):

        joblib.dump(
            pipeline,
            self.config.preprocessor_obj_file_path
        )

    logger.info("Preprocessor object saved")

    ## Saving Transformation Report 
    def save_transformation_report(self):

        report_path = self.config.TRANSFORMATION_REPORT

        with open(report_path, "w") as f:

            json.dump(self.transformation_report, f, indent=4)

        logger.info(f"Transformation report saved at {report_path}")


    ## Initialising Data Transformation Pipeline 

    def initiate_data_transformation(self):
        try: 
            
            ## Intiating Data Transformation stage

            self.transformation_report["stage_metadata"] = {
                "stage_name" : "Data Transformation",
                "stage_status" : "Running",
                "start_time" : str(datetime.now())
            }

            logger.info("Initiating Data Transformation Stage.")

            ## Saving start_time
            start_time = datetime.now()

            ## Loading Dataset
            self.load_data()

            ## Save country names for later use and drop it 
            self.separate_country_column()

            ## Clean currency from columns for transformation and update the datatype from object to numeric
            self.convert_object_to_numeric()      

            ## Create Preprocessor Pipeline with Knn Imputer, Power transformer and Rosbust scaling with PCA
            self.transform_data()   

            transformed_data, pipeline = self.transform_data()

            print("Total NaNs:", np.isnan(transformed_data).sum())

            ## Save Transformed Dataset
            self.save_transformed_data(transformed_data)

            ## Save Preprocessor pickle file
            self.save_preprocessor(pipeline)

            ## Stage Success
            self.transformation_report["stage_metadata"]["stage_status"] = "Success"
            self.transformation_report["stage_metadata"]["end_time"] = str(datetime.now())

            ## Calculating Stage Duration
             
            stage_duration = (datetime.now() - start_time).total_seconds()
            self.transformation_report["stage_metadata"]["stage_duration"] = stage_duration
            
            # Saving Transformations Report JSON
            self.save_transformation_report()

            logger.info("All Data Transformation Checks Completed Successfully.")

        except Exception as e:
            
            logger.error(f"Transformation Error : {e}")

            end_time = datetime.now()

            self.transformation_report["stage_metadata"]["stage_status"] = "Failed"
            self.transformation_report["stage_metadata"]["end_time"] = str(end_time)
            self.transformation_report["stage_metadata"]["error"] = str(e)

            self.save_transformation_report()

            raise e


        

[2026-04-01 15:24:17,387: INFO: 1371372621: Preprocessor object saved]


In [11]:
# Pipeline
## Data Transformation Pipeline
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.initiate_data_transformation()
    logger.info(f"Data Transformation Pipeline Completed Successfully.")
except Exception as e:
    logger.error(f"Data Trasforamation Pipeline Failed: {e}")
    raise e

[2026-04-01 15:24:28,636: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-01 15:24:28,650: INFO: common: yaml file: params.yaml loaded successfully]


[2026-04-01 15:24:28,662: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-04-01 15:24:28,665: INFO: common: created directory at: artifacts]
[2026-04-01 15:24:28,668: INFO: common: created directory at: artifacts/data_transformation]
[2026-04-01 15:24:28,669: INFO: 1371372621: Initiating Data Transformation Stage.]
[2026-04-01 15:24:31,297: INFO: 1371372621: Data Loaded Succesfully.]
[2026-04-01 15:24:31,298: INFO: 1371372621: Country column separated.]
[2026-04-01 15:24:31,298: INFO: 1371372621: Cleaning and validating the data before preprocessing.]
[2026-04-01 15:24:31,361: INFO: 1371372621: Object columns converted to numeric.]
[2026-04-01 15:24:31,365: INFO: 1371372621: Total missing values after cleaning: 11740]
[2026-04-01 15:24:31,365: INFO: 1371372621: Data types after conversion:
birth_rate                float64
business_tax_rate         float64
co2_emissions             float64
days_to_start_business    float64
ease_of_business          float64
energy_usage 